# 1. Where are the cells in a jet?

I'll take one jet from a simulated long-lived particle (LLP) decay, draw its cells and see if I get Emil's timing score. I use the **PPP first look** kernel.

In [ ]:
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display
from matplotlib.patches import Circle

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent

llp_folder = project / "DataForEmre/collision/LLP/N1N2_rpv_1400_10ns"
llp_files = sorted(llp_folder.glob("*.root"))
assert llp_files, f"No LLP ROOT files in {llp_folder}"
llp_file = llp_files[0]

## Which jets do I keep?

An LLP event can also contain other jets. I start with `jet_is_llp_matched == 1`, the match stored in the simulation. Awkward keeps one list of jets per event.

I ask for $p_T\geq30$ GeV (the file gives MeV), $|\eta|<1.7$ and energy in both Tile and liquid argon. Each row in the table shows how many jets that cut removes from the row above.

In [ ]:
fields = ["jet_is_llp_matched", "jet_pt", "jet_eta", "jet_time_ATLAS", "jet_TileEnergyFrac"]
with uproot.open(llp_file) as source:
    jets = source["JetContent"].arrays(fields, library="ak")

matched = jets["jet_is_llp_matched"] == 1
pt = jets["jet_pt"]
eta = jets["jet_eta"]
time = jets["jet_time_ATLAS"]
tile = jets["jet_TileEnergyFrac"]

def valid_number(values):
    return np.isfinite(values) & (values != -999999)

In [ ]:
keep = ak.ones_like(pt, dtype=bool)
counts = {"All jets": int(ak.sum(keep))}
cuts = {
    "LLP matched": matched,
    "Valid momentum, direction and time": valid_number(pt) & valid_number(eta) & valid_number(time),
    "pT ≥ 30 GeV": pt / 1000 >= 30,
    "|eta| < 1.7": abs(eta) < 1.7,
    "Energy in Tile and LAr": (tile > 0) & (tile < 1),
}
for name, passes in cuts.items():
    keep = keep & passes
    counts[name] = int(ak.sum(keep))
cutflow = pd.DataFrame({"Jets left": counts})
cutflow["Removed at this step"] = -cutflow["Jets left"].diff().fillna(0).astype(int)
display(cutflow)

## Take one jet apart

I pick the first LLP-matched jet. Its cells are in one long list for the whole event. To find my part of the list, I count clusters per jet and cells per cluster.

For example, say two jets have `[2, 1]` clusters, with `[3, 2, 4]` cells per cluster. The first jet gets the first five cells and the second gets the next four.

In [ ]:
rows = ak.where(ak.any(matched, axis=1))[0]
assert len(rows), "No matched LLP jet in the first file."
event_index = int(rows[0])
jet_index = int(ak.where(matched[event_index])[0][0])

fields = [
    "runNumber", "eventNumber", "jet_n_clusters", "cluster_n_cells",
    "jet_eta", "jet_phi", "cell_eta", "cell_phi", "jet_llp_idx", "llp_daughter_parent",
    "llp_daughter_calo_eta", "llp_daughter_calo_phi",
    "jet_llp_n_aligned_cells", "jet_llp_n_aligned_clusters",
    "cell_time", "cell_time_sigma", "cell_x", "cell_y", "cell_z", "jet_BIB_score",
]
with uproot.open(llp_file) as source:
    event = source["JetContent"].arrays(
        fields, entry_start=event_index, entry_stop=event_index + 1, library="ak",
    )[0]

print(f"Run {event['runNumber']}, event {event['eventNumber']}, jet {jet_index}")

In [ ]:
cluster_counts = event["jet_n_clusters"]
cell_counts = event["cluster_n_cells"]
cluster_start = int(ak.sum(cluster_counts[:jet_index]))
cluster_stop = cluster_start + int(cluster_counts[jet_index])
cell_start = int(ak.sum(cell_counts[:cluster_start]))
cell_stop = int(ak.sum(cell_counts[:cluster_stop]))
cell_slice = slice(cell_start, cell_stop)

assert int(ak.sum(cluster_counts)) == len(cell_counts)
assert int(ak.sum(cell_counts)) == len(event["cell_eta"])
print(f"Clusters [{cluster_start}:{cluster_stop}], cells [{cell_start}:{cell_stop}]")

## Where should the decay particles hit?

I centre the plot on the jet. Blue dots are cells. Red crosses are where the simulation expects the LLP decay particles to hit. $\eta$ tells me how close a direction is to the beam and $\phi$ is the angle around it. I wrap $\Delta\phi$ because $-\pi$ and $\pi$ are the same direction.

In the zoom, a cell inside any dashed circle is aligned with a decay particle. The radius is $\Delta R=\sqrt{(\Delta\eta)^2+(\Delta\phi)^2}=0.08$.

In [ ]:
def wrap_phi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

axis_eta = float(event["jet_eta"][jet_index])
axis_phi = float(event["jet_phi"][jet_index])
cell_eta = ak.to_numpy(event["cell_eta"][cell_slice]) - axis_eta
cell_phi = wrap_phi(ak.to_numpy(event["cell_phi"][cell_slice]) - axis_phi)

parent = int(event["jet_llp_idx"][jet_index])
assert parent >= 0, "This jet has no assigned LLP."
daughters = event["llp_daughter_parent"] == parent
d_eta = ak.to_numpy(event["llp_daughter_calo_eta"][daughters])
d_phi = ak.to_numpy(event["llp_daughter_calo_phi"][daughters])
assert len(d_eta) and np.all(valid_number(d_eta) & valid_number(d_phi))
daughter_eta = d_eta - axis_eta
daughter_phi = wrap_phi(d_phi - axis_phi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax in axes:
    ax.scatter(cell_eta, cell_phi, s=14, color="tab:blue", label="Jet cells")
    ax.scatter(daughter_eta, daughter_phi, c="tab:red", marker="x", label="Projected daughters")
    ax.scatter(0, 0, c="black", marker="+", label="Jet axis")
    for eta_hit, phi_hit in zip(daughter_eta, daughter_phi):
        ax.add_patch(Circle((eta_hit, phi_hit), 0.08, fill=False, color="tab:red", linestyle="--"))
    ax.set(xlabel=r"$\Delta\eta$", ylabel=r"$\Delta\phi$ [rad]", aspect="equal")
    ax.grid(alpha=0.2)

axes[0].set_title("Jet cells and projected daughters")
axes[1].set(title="Zoom around the jet axis", xlim=(-0.45, 0.45), ylim=(-0.45, 0.45))
axes[1].legend(fontsize=8)
plt.show()

Here **10 of 36 cells** fall inside a circle. I count the clusters with at least 30% aligned cells and compare both counts with the file.

The code I checked also asks for 20% aligned cells across the jet. I'd ask Emil which version produced these files. [Emil's matching code](https://gitlab.cern.ch/ezaya/athjetconstituentsalg-public/-/blob/a7c625c364893bbc90248fbb98a5251c1a54fdeb/AthJetConstituents/src/AthProcessing.cxx)

In [ ]:
delta_eta = cell_eta[:, None] - daughter_eta
delta_phi = wrap_phi(cell_phi[:, None] - daughter_phi)
distance_sq = delta_eta**2 + delta_phi**2
aligned = (distance_sq < 0.08**2).any(axis=1)
sizes = cell_counts[cluster_start:cluster_stop]
aligned_per_cluster = ak.sum(ak.unflatten(aligned, sizes), axis=1)
aligned_clusters = int(ak.sum(aligned_per_cluster / sizes >= 0.30))

assert int(aligned.sum()) == int(event["jet_llp_n_aligned_cells"][jet_index])
assert aligned_clusters == int(event["jet_llp_n_aligned_clusters"][jet_index])
print(f"Aligned cells: {aligned.sum()} / {len(aligned)}. Aligned clusters: {aligned_clusters}.")

## Why can BIB look late?

ATLAS subtracts the time light would take from the collision point to cell $i$. A slow LLP can decay on the way, giving several cells a common delay. A particle from beam-induced background (BIB) can cross the detector roughly parallel to the beam. For a cell at radius $R_i=\sqrt{x_i^2+y_i^2}$ and beam coordinate $z_i$, I use Simone Gallareto’s flat and beam timing models.

$$
t_i^{\rm flat}=a, \qquad t_i^{\rm beam}=g_i+b, \qquad
g_i=\frac{s z_i-\sqrt{R_i^2+z_i^2}}{c}.
$$

Here $s=\pm1$ gives the beam direction. The term $s z_i/c$ is the travel time along the beam relative to passing $z=0$. The square-root term is the flight time ATLAS subtracts. The constants $a$ and $b$ shift the whole pattern in time.

For example, at $R=1500$ mm and $z=0$, $g\approx-5$ ns. If an early hit gets assigned to the previous bunch crossing, its time increases by $25$ ns relative to that crossing. It looks **20 ns late**. The pattern across cells is still there. [Simone, §§7.1–7.2.1, pp. 40–42](https://kth.diva-portal.org/smash/get/diva2%3A1985126/FULLTEXT01.pdf#page=45)

### How Simone compared the shapes

Simone fitted the same $N$ cells in one cluster twice. Each shape gets one fitted time offset. For measured time $t_i$, predicted time $f_i$ and cell timing uncertainty $\sigma_i$, the comparison uses $\chi^2_{\rm red}$. The smaller value means the measured times lie closer to that shape in units of their uncertainty.

$$
\chi^2_{\rm red}=\frac{1}{N-1}\sum_{i=1}^{N}
\left(\frac{t_i-f_i}{\sigma_i}\right)^2 .
$$

The $N-1$ accounts for the fitted offset. Simone chose the beam sign using the times at the two ends of the cluster in $z$. The study used clusters with $p_T>10$ GeV and at least six cells with $p_T>500$ MeV each. The beam fit won for 5,070 of 6,721 background clusters in the loss-map data. The flat fit won for 107,191 of 132,266 simulated signal clusters. These counts are per **cluster**. [Method, §§7.2.4–7.2.5, pp. 46–47](https://kth.diva-portal.org/smash/get/diva2%3A1985126/FULLTEXT01.pdf#page=51) · [Results, §8.4.1, p. 52](https://kth.diva-portal.org/smash/get/diva2%3A1985126/FULLTEXT01.pdf#page=57)

### How I get Emil's score for one jet

Emil compares the shapes across the usable cells of the **whole jet**. I use the stored $\sigma_i$ and weights $w_i=1/\sigma_i^2$, so a precise time counts more. I set the beam direction from the sign of the weighted time–$z$ covariance. The code below uses mm and ns. For two cells the common offset cancels in $t_i-t_j$. The flat fit predicts a difference of zero. The beam fit predicts $g_i-g_j$. This is why cell pairs can help even when two jets have the same average time.

In [ ]:
fields = ["cell_time", "cell_time_sigma", "cell_x", "cell_y", "cell_z"]
raw = np.column_stack([ak.to_numpy(event[name][cell_slice]) for name in fields])
usable = valid_number(raw).all(axis=1) & (raw[:, 1] > 0)
t, sigma, x, y, z = raw[usable].T
assert len(t) >= 4, "The timing fit needs at least four valid cells."
weights = 1 / sigma**2

def weighted_mean(values):
    return np.average(values, weights=weights)

covariance = weighted_mean((t - weighted_mean(t)) * (z - weighted_mean(z)))
direction = np.sign(covariance)
if direction == 0:
    direction = 1 if weighted_mean(z) >= 0 else -1
distance_mm = np.hypot(np.hypot(x, y), z)
c_mm_per_ns = 299.792458
beam = (direction * z - distance_mm) / c_mm_per_ns

For either shape, I subtract $g_i$ first ($g_i=0$ for the flat fit). The best offset $\hat b_g$ is the weighted mean of what's left. $V_g$ measures how much the cell times still vary around the fitted shape.

$$
\hat b_g=\frac{\sum_i w_i(t_i-g_i)}{\sum_i w_i},
\qquad V_g=\frac{\sum_i w_i(t_i-g_i-\hat b_g)^2}{\sum_i w_i}.
$$

The numerator of $V_g$ is the fitted $\chi^2$. Emil combines the two spreads into a score between 0 and 1.

$$
S=\frac{V_{\rm flat}}{V_{\rm flat}+V_{\rm beam}}.
$$

If the beam shape fits better, $V_{\rm beam}$ is smaller and $S>0.5$. In notebook 03 I choose the tagging cut from validation data.

In [ ]:
def residual_variance(template):
    residual = t - template
    return weighted_mean((residual - weighted_mean(residual))**2)

flat_var = residual_variance(0)
beam_var = residual_variance(beam)
score = flat_var / (flat_var + beam_var)
stored = float(event["jet_BIB_score"][jet_index])
np.testing.assert_allclose(score, stored, rtol=0, atol=1e-7)
print(f"Calculated score: {score:.5f}. Stored score: {stored:.5f}.")

I get **0.21440**, the same score as in the file. The flat shape fits this jet better. In [notebook 03](03_train_and_compare.ipynb) I compare the score with ParT.